Q5: Evaluation Metrics from a Multi-Class Confusion Matrix
The system classified 90 animals into Cat, Dog, or Rabbit. The results are shown below:
System \ Gold	Cat	Dog	Rabbit
Cat	5	10	5
Dog	15	20	10
Rabbit	0	15	10

3.	Programming Implementation
Write Python code that:
    1.	Accepts the confusion matrix above as input.
    2.	Computes per-class precision and recall.
    3.	Computes macro-averaged and micro-averaged precision and recall.
    4.	Prints all results clearly.


In [2]:
import numpy as np

# Confusion matrix [rows=predicted, cols=gold/true]
cm = np.array([
    [5, 10, 5],   # Predicted Cat
    [15, 20, 10], # Predicted Dog
    [0, 15, 10]   # Predicted Rabbit
])

classes = ['Cat', 'Dog', 'Rabbit']

# Per-class metrics
precisions = []
recalls = []

for i in range(3):
    # Precision: TP / (TP + FP) = diagonal / column sum
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    precision = tp / (tp + fp)
    precisions.append(precision)

    # Recall: TP / (TP + FN) = diagonal / row sum
    fn = cm[i, :].sum() - tp
    recall = tp / (tp + fn)
    recalls.append(recall)

print("Per-Class Metrics:")
for i, cls in enumerate(classes):
    print(f"{cls}: Precision={precisions[i]:.3f}, Recall={recalls[i]:.3f}")

# Macro averaging
macro_precision = np.mean(precisions)
macro_recall = np.mean(recalls)
print(f"\nMacro Precision: {macro_precision:.3f}")
print(f"Macro Recall: {macro_recall:.3f}")

# Micro averaging (total TP / total predictions)
total_tp = cm.trace()
micro_precision = total_tp / cm.sum()
micro_recall = total_tp / cm.sum()
print(f"Micro Precision: {micro_precision:.3f}")
print(f"Micro Recall: {micro_recall:.3f}")


Per-Class Metrics:
Cat: Precision=0.250, Recall=0.250
Dog: Precision=0.444, Recall=0.444
Rabbit: Precision=0.400, Recall=0.400

Macro Precision: 0.365
Macro Recall: 0.365
Micro Precision: 0.389
Micro Recall: 0.389


Q1. Programming: Bigram Language Model Implementation (based on “Activity: I love NLP corpus” slide)

Tasks:
Write a Python program to:
    1.	Read the training corpus:
    2.	<s> I love NLP </s>
    3.	<s> I love deep learning </s>
    4.	<s> deep learning is fun </s>
    5.	Compute unigram and bigram counts.
    6.	Estimate bigram probabilities using MLE.
    7.	Implement a function that calculates the probability of any given sentence.
    8.	Test your function on both sentences:
        o	<s> I love NLP </s>
        o	<s> I love deep learning </s>
    9.	Print which sentence the model prefers and why.


In [3]:
from collections import defaultdict, Counter
import re
import math

def preprocess_sentence(sentence):
    """Add <s> and </s> tokens and split into words."""
    return ['<s>'] + sentence.split() + ['</s>']

def compute_ngram_counts(corpus):
    """Compute unigram and bigram counts from tokenized corpus."""
    unigram_counts = Counter()
    bigram_counts = defaultdict(Counter)

    for sentence in corpus:
        for token in sentence:
            unigram_counts[token] += 1

        for i in range(len(sentence) - 1):
            prev, next_token = sentence[i], sentence[i+1]
            bigram_counts[prev][next_token] += 1

    return unigram_counts, bigram_counts

def sentence_probability(sentence, bigram_counts, unigram_counts):
    """Compute MLE bigram probability of a sentence."""
    tokens = preprocess_sentence(sentence)
    prob = 1.0

    for i in range(len(tokens) - 1):
        prev, next_token = tokens[i], tokens[i+1]
        count_prev = unigram_counts[prev]
        count_bigram = bigram_counts[prev][next_token]
        prob_bigram = count_bigram / count_prev if count_prev > 0 else 0
        prob *= prob_bigram

    return prob, math.log(prob, 2) if prob > 0 else float('-inf')

# Training corpus
training_corpus = [
    "<s> I love NLP </s>",
    "<s> I love deep learning </s>",
    "<s> deep learning is fun </s>"
]

# 1. Tokenize corpus
tokenized_corpus = [preprocess_sentence(sent) for sent in training_corpus]
print("Tokenized corpus:")
for sent in tokenized_corpus:
    print(sent)
print()

# 2. Compute counts
unigram_counts, bigram_counts = compute_ngram_counts(tokenized_corpus)

print("Unigram counts:")
for word, count in unigram_counts.most_common():
    print(f"  {word}: {count}")

print("\nBigram counts:")
for prev in sorted(bigram_counts.keys()):
    print(f"  {prev}: {dict(bigram_counts[prev])}")
print()

# 3. Test sentences
test_sentences = [
    "<s> I love NLP </s>",
    "<s> I love deep learning </s>"
]

print("Sentence probabilities:")
results = []
for sentence in test_sentences:
    prob, log_prob = sentence_probability(sentence, bigram_counts, unigram_counts)
    result = {'sentence': sentence, 'prob': prob, 'log_prob': log_prob}
    results.append(result)
    print(f"  P({sentence}) = {prob:.6f} (log2 = {log_prob:.3f})")

# 4. Which is preferred?
s1, s2 = results
if s1['prob'] > s2['prob']:
    preferred = s1
    other = s2
    reason = f"S1 has higher probability ({s1['prob']:.6f} > {s2['prob']:.6f})"
elif s2['prob'] > s1['prob']:
    preferred = s2
    other = s1
    reason = f"S2 has higher probability ({s2['prob']:.6f} > {s1['prob']:.6f})"
else:
    preferred = s1
    reason = "Both sentences have equal probability"

print(f"\nModel prefers: {preferred['sentence']}")
print(f"Why: {reason}")


Tokenized corpus:
['<s>', '<s>', 'I', 'love', 'NLP', '</s>', '</s>']
['<s>', '<s>', 'I', 'love', 'deep', 'learning', '</s>', '</s>']
['<s>', '<s>', 'deep', 'learning', 'is', 'fun', '</s>', '</s>']

Unigram counts:
  <s>: 6
  </s>: 6
  I: 2
  love: 2
  deep: 2
  learning: 2
  NLP: 1
  is: 1
  fun: 1

Bigram counts:
  </s>: {'</s>': 3}
  <s>: {'<s>': 3, 'I': 2, 'deep': 1}
  I: {'love': 2}
  NLP: {'</s>': 1}
  deep: {'learning': 2}
  fun: {'</s>': 1}
  is: {'fun': 1}
  learning: {'</s>': 1, 'is': 1}
  love: {'NLP': 1, 'deep': 1}

Sentence probabilities:
  P(<s> I love NLP </s>) = 0.041667 (log2 = -4.585)
  P(<s> I love deep learning </s>) = 0.020833 (log2 = -5.585)

Model prefers: <s> I love NLP </s>
Why: S1 has higher probability (0.041667 > 0.020833)
